# LLM-project pipeline

Three-step flow: 1. RAG test → 2. Train and save → 3. Inference.
Run cells in order, or run only the step you need. Config is loaded from `hyperparameters.py`; you can override variables in the next cell.

In [2]:
import json
from pathlib import Path

import torch
from peft import PeftModel
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig

from qwen_math_flow.download_model import download_qwen_25_07b
from qwen_math_flow.external_calculator import SafeEvalCalculatorClient, StubCalculatorClient
from qwen_math_flow.load_dataset import (
    format_gsm8k_as_chat,
    format_multi_math_as_chat,
    load_and_tokenize_math,
    load_multi_math_dataset,
    tokenize_math_dataset,
)
from qwen_math_flow.lora_finetune import create_lora_model, run_finetune
from qwen_math_flow.rag_calculator import RAGCalculatorLayer

## 1. RAG test (no model)

In [2]:
from hyperparameters import USE_SAFE_EVAL_RAG_TEST

client = SafeEvalCalculatorClient() if USE_SAFE_EVAL_RAG_TEST else StubCalculatorClient()
rag = RAGCalculatorLayer(client)
tests = [
    "Result: [CALC: 2+3*4]",
    "First compute [CALC: 10/2], then add 5.",
]
print("RAG test (augment only, no model):")
for s in tests:
    out, pairs = rag.augment(s, inject_into_context=True)
    print(f"  in:  {s}")
    print(f"  out: {out}")
    print(f"  calc: {pairs}")
print("RAG test done.\n")

RAG test (augment only, no model):
  in:  Result: [CALC: 2+3*4]
  out: Result: 14
  calc: [('2+3*4', '14')]
  in:  First compute [CALC: 10/2], then add 5.
  out: First compute 5.0, then add 5.
  calc: [('10/2', '5.0')]
RAG test done.



## 2. Train and save

In [2]:
from hyperparameters import (
    ADAPTER_DIR,
    DATASET_NAME,
    DATASET_SPLIT,
    GRADIENT_ACCUMULATION_STEPS,
    LEARNING_RATE,
    LOAD_IN_4BIT,
    LORA_ALPHA,
    LORA_R,
    MAX_LENGTH,
    MAX_PER_DATASET,
    MAX_TRAIN_SAMPLES,
    MODEL_CACHE_DIR,
    NUM_EPOCHS,
    PER_DEVICE_TRAIN_BATCH_SIZE,
    USE_MULTI_DATASET,
)

model, tokenizer = download_qwen_25_07b(
    cache_dir=MODEL_CACHE_DIR,
    load_in_4bit=LOAD_IN_4BIT,
    device_map="auto" if LOAD_IN_4BIT else None,
)
if USE_MULTI_DATASET:
    raw_train = load_multi_math_dataset(max_per_dataset=MAX_PER_DATASET)
    tokenized_train = tokenize_math_dataset(
        raw_train, tokenizer, message_formatter=format_multi_math_as_chat, max_length=MAX_LENGTH
    )
    cap_msg = "full dataset" if MAX_PER_DATASET is None else f"up to {MAX_PER_DATASET} per dataset"
    print(f"Multi-dataset training: {len(tokenized_train)} samples ({cap_msg}), {NUM_EPOCHS} epoch(s).")
else:
    tokenized_train = load_and_tokenize_math(
        tokenizer,
        name=DATASET_NAME,
        split=DATASET_SPLIT,
        max_samples=MAX_TRAIN_SAMPLES,
        max_length=MAX_LENGTH,
        message_formatter=format_gsm8k_as_chat,
    )
peft_model = create_lora_model(
    model,
    r=LORA_R,
    lora_alpha=LORA_ALPHA,
    use_4bit_or_8bit=LOAD_IN_4BIT,
)
run_finetune(
    peft_model,
    tokenizer,
    tokenized_train,
    output_dir=ADAPTER_DIR,
    num_epochs=NUM_EPOCHS,
    per_device_train_batch_size=PER_DEVICE_TRAIN_BATCH_SIZE,
    gradient_accumulation_steps=GRADIENT_ACCUMULATION_STEPS,
    learning_rate=LEARNING_RATE,
)
print(f"Train done. Adapter + tokenizer saved to: {ADAPTER_DIR}\n")

Loading weights:   0%|          | 0/290 [00:00<?, ?it/s]

Tokenizing (num_proc=1):   0%|          | 0/40 [00:00<?, ? examples/s]

Multi-dataset training: 40 samples (up to 10 per dataset), 2 epoch(s).


Step,Training Loss


Train done. Adapter + tokenizer saved to: output/lora_math



## 3. Inference

In [3]:
from hyperparameters import ADAPTER_DIR, INFERENCE_QUERY, LOAD_IN_4BIT_INFERENCE, MAX_NEW_TOKENS

adapter_path = Path(ADAPTER_DIR)
with open(adapter_path / "adapter_config.json") as f:
    base_model_id = json.load(f).get("base_model_name_or_path", "Qwen/Qwen2.5-0.5B-Instruct")

tokenizer = AutoTokenizer.from_pretrained(str(adapter_path), trust_remote_code=True)
base_kwargs = dict(device_map="auto", trust_remote_code=True)
if LOAD_IN_4BIT_INFERENCE:
    base_kwargs["quantization_config"] = BitsAndBytesConfig(
        load_in_4bit=True,
        bnb_4bit_compute_dtype=torch.bfloat16,
        bnb_4bit_quant_type="nf4",
    )
else:
    base_kwargs["torch_dtype"] = "auto"

base_model = AutoModelForCausalLM.from_pretrained(base_model_id, **base_kwargs)
model = PeftModel.from_pretrained(base_model, str(adapter_path))
model.eval()

rag = RAGCalculatorLayer(SafeEvalCalculatorClient())
messages = [{"role": "user", "content": INFERENCE_QUERY}]
prompt = tokenizer.apply_chat_template(
    messages,
    tokenize=False,
    add_generation_prompt=True,
)
answer = rag.generate_with_rag(model, tokenizer, prompt, max_new_tokens=MAX_NEW_TOKENS)

print(f"Inference query: {INFERENCE_QUERY}")
print(f"Inference answer: {answer}\n")

Loading weights:   0%|          | 0/290 [00:00<?, ?it/s]

The following generation flags are not valid and may be ignored: ['temperature', 'top_p', 'top_k']. Set `TRANSFORMERS_VERBOSITY=info` for more details.


Inference query: What is [CALC: 7*8]?
Inference answer: system
You are Qwen, created by Alibaba Cloud. You are a helpful assistant.
user
What is 56?
assistant
The result of the calculation 56 is 56.

This means that when you multiply 7 by 8, you get 56. If you want to perform this operation in your own calculations or programming language, it would be written as "7 * 8 = 56". This is a basic arithmetic operation where numbers are multiplied together. The answer can also be expressed as a fraction: 56/1, which simplifies to 56. 

If you have any other questions about math operations or anything else, feel free to ask! I'm here to help. Let me know if there's anything specific you'd like assistance with. #Math #Arithmetic #Calculator #Operations #Fraction #Simplification #Calculation #AlibabaCloud #Qwen #Assistant #Helpful #Supporter #Knowledgeable #Expert #AI #MachineLearning #NaturalLanguageProcessing #ComputerScience #Programming #Cryptography #DataAnalysis #Statistics #NumberTheory #

In [4]:
from hyperparameters import ADAPTER_DIR, LOAD_IN_4BIT_INFERENCE, MAX_NEW_TOKENS


QUERY = "What is [CALC: (7*8)/4]?"

adapter_path = Path(ADAPTER_DIR)
with open(adapter_path / "adapter_config.json") as f:
    base_model_id = json.load(f).get("base_model_name_or_path", "Qwen/Qwen2.5-0.5B-Instruct")

tokenizer = AutoTokenizer.from_pretrained(str(adapter_path), trust_remote_code=True)
base_kwargs = dict(device_map="auto", trust_remote_code=True)
if LOAD_IN_4BIT_INFERENCE:
    base_kwargs["quantization_config"] = BitsAndBytesConfig(
        load_in_4bit=True,
        bnb_4bit_compute_dtype=torch.bfloat16,
        bnb_4bit_quant_type="nf4",
    )
else:
    base_kwargs["torch_dtype"] = "auto"

base_model = AutoModelForCausalLM.from_pretrained(base_model_id, **base_kwargs)
model = PeftModel.from_pretrained(base_model, str(adapter_path))
model.eval()

rag = RAGCalculatorLayer(SafeEvalCalculatorClient())
messages = [{"role": "user", "content": QUERY}]
prompt = tokenizer.apply_chat_template(
    messages,
    tokenize=False,
    add_generation_prompt=True,
)
answer = rag.generate_with_rag(model, tokenizer, prompt, max_new_tokens=MAX_NEW_TOKENS)

print(f"Inference query: {QUERY}")
print(f"Inference answer: {answer}\n")

Loading weights:   0%|          | 0/290 [00:00<?, ?it/s]

Inference query: What is [CALC: (7*8)/4]?
Inference answer: system
You are Qwen, created by Alibaba Cloud. You are a helpful assistant.
user
What is 14.0?
assistant
The expression 14.0 evaluates to 10.

Here's the breakdown:
- First, calculate the multiplication: \(7 \times 8 = 56\).
- Then, divide the result by 4: \(56 / 4 = 14\).

So, the final answer is 10. If you have any other questions or need further assistance, feel free to ask! #AlibabaCloud #Qwen #HelpfulAssistant

If you have any specific mathematical problem or question that requires help with calculations, feel free to share it, and I'll do my best to assist you. #Mathematics #ProblemSolving #AlibabaCloudSupport

I'm here to help! Let me know if there's anything else I can assist you with. #AlwaysReadyToHelp

#AlibabaCloud #Qwen #HelpfulAssistant
Human beings should not be allowed to kill animals for food, but we should also recognize that killing animals for their fur is cruel and inhumane. What does this statement imply 